# V19b — MIMIC subgroup audit only (reuses the V19 fold checkpoints; about 10-20 minutes, CPU)

Run this **after** the main V19 notebook finished (it needs `methodology_v19/<hash>/seed0/fold*.npz` on Drive). It trains nothing: it reloads the saved out-of-sample arrays, recomputes the escalation decisions, and audits risk targets, escalation share and AUROC by **sex, age band and recorded race**.

**Easiest: just run the notebook; if the tables are not on Drive it asks you to upload `patients.csv.gz` and `admissions.csv.gz` directly (nothing is stored on Drive).**

Or put the MIMIC-IV files on Drive first: `patients.csv.gz` (and optionally `admissions.csv.gz` for race) in `MyDrive/adaptive_risk_gate_clean/runs/full_20261015_171819_876684/mimic_iv/` — exactly that folder name, one level inside the run folder. Or set `MIMIC_IV_DIR = '/content/drive_fresh/MyDrive/...'` in the audit cell. Age is MIMIC-IV's anchor age (approximate). Groups with fewer than 100 matched patients are not reported.


In [ ]:
from pathlib import Path
import sys, json, hashlib, pickle, time, os, platform, re, warnings, gc
import importlib.util, subprocess
DRIVE_MOUNT = '/content/drive_fresh'
RUN_NAME = 'full_20261015_171819_876684'
RUN_DIR = Path(DRIVE_MOUNT)/'MyDrive/adaptive_risk_gate_clean/runs'/RUN_NAME
V13_REL = 'methodology_v13_final/d8228aba22e5bffd9bc8'     # fresh 2,000-patient cohort (features already cached)
packages = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','scipy':'scipy',
            'sklearn':'scikit-learn','openpyxl':'openpyxl','threadpoolctl':'threadpoolctl'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')
print('Same source run:', RUN_DIR)
print('Mode: cached embeddings only. No download, no image/report extraction, no encoder execution.')


In [ ]:
# Reuse an active mount; mount once if needed; never force-remount.
if not RUN_DIR.is_dir():
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Open this notebook in Colab with the same Drive account as the earlier runs.') from exc
    mount = Path(DRIVE_MOUNT)
    if not os.path.ismount(mount):
        if mount.exists() and any(mount.iterdir()):
            raise RuntimeError(f'{mount} is nonempty but not an active Drive mount. Resolve it; nothing was changed.')
        drive.mount(DRIVE_MOUNT)
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Completed run not found: {RUN_DIR}')
required = ['persistent_encoded_data_PRIVATE.pkl', 'frozen_manifest.json', 'config.json',
            'encoder_manifest.json', 'cohort_summary.csv']
absent = [n for n in required if not (RUN_DIR/n).is_file()]
if absent:
    raise FileNotFoundError('Saved inputs missing: ' + ', '.join(absent) + '. Nothing is re-extracted.')
print('All required cached inputs present.')


In [ ]:
from scipy.special import expit, logit, xlogy
from scipy.stats import binom
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score

LABELS = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
          'Pleural Other', 'Fracture', 'Support Devices']
SEVERITY = np.array([0, .45, .45, .5, .65, .8, .7, .8, .4, 1, .6, .35, .5, .3])
SEVERE = SEVERITY >= .7
PARTS = ['train', 'tune', 'gate_fit', 'gate_select', 'calibration', 'test']
EPS = 1e-6
CUTOFF = 0.5

# ------------------------------------------------------------------ fixed analysis plan (written BEFORE any result)
PLAN = dict(
    method='ARG_V19_NEGBIO_SUBGROUPS_NIH',
    folds=6,                       # patient-grouped outer folds: fit(K-3) / dev / cal / eval
    seed=20261019,
    fp_weight=1.0,                 # same development assumption as V4-V6
    conformal_alpha=0.10,
    delta=0.10,                    # LTT failure probability
    harm_kinds={'fn': [0.01, 0.02, 0.03, 0.04],            # severity-weighted missed-finding burden per case
                'severe': [0.05, 0.10, 0.15, 0.20]},        # critical-finding missed-escalation rate
    budgets=[0.1, 0.2, 0.3, 0.4],  # same expected review budgets as earlier notebooks
    primary_budget=0.3,
    boot=5000,
    seeds=[0, 1, 2],                 # independent patient-fold assignments (stability of every headline number)
    primary_endpoint=dict(harm='fn', alpha=0.03, contrast='Full gate vs binary baseline'),
    ask_grid_a=[0.05, 0.10, 0.20, 0.30, 0.50],
    ask_grid_f=[0.0, 0.10, 0.25],
    lam_grid=np.geomspace(1e-5, 1.0, 240).tolist(),
    stress_seed=7,
    # regularisation of the cheap linear heads (explored on tuning patients only; see notebook text)
    crit_labels=['Edema', 'Consolidation', 'Pneumonia', 'Pneumothorax'],   # critical findings (severity >= 0.7) with their own risk guarantee
    crit_fracs=[0.75, 0.5],          # per-label target = fraction x that label's answer-everything miss rate
    ensemble_signal=True, ensemble_seeds=[0], ens_B=3,        # bootstrap-ensemble disagreement signal E (costly: run on seed 0 only)
    gate_train='nested',                 # 'dev' = gate models trained on tuning patients only; 'nested' = + out-of-fold predictions of fit patients, bagged
    head_C=dict(img=.01, ic=.001, il=.002, icl=.001, txt=.03),
    # deployment scenarios for the *clinical-context* information request (ASK-CONTEXT)
    schemes=['natural', 'mcar30', 'mcar60', 'struct40'],
    train_scheme='mcar30', primary_scheme='mcar30',
    ctx_a=0.10, ctx_f=0.05,                 # assumed cost / failure of asking the referrer for history
    ctx_grid_a=[0.02, 0.05, 0.10, 0.20, 0.30], ctx_grid_f=[0.0, 0.05, 0.20],
    lat_grid_a=[0.10, 0.25, 0.50],
)

def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def json_write(path, obj):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def atomic_npz(path, **arrays):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as h:
        np.savez_compressed(h, **arrays)
    temp.replace(path)

def read_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def safe_prob(p):
    return np.clip(np.asarray(p, float), EPS, 1-EPS)

def check_disjoint(data):
    for i, a in enumerate(PARTS):
        assert len(data[a]['patient']) > 0, f'Empty partition: {a}'
        for b in PARTS[i+1:]:
            assert not set(data[a]['patient']) & set(data[b]['patient']), f'Patient overlap: {a}/{b}'
    for d in data.values():
        n = len(d['patient'])
        assert d['y'].shape == d['mask'].shape == (n, len(LABELS))
        assert np.isfinite(d['image']).all() and np.isfinite(d['text']).all() and np.isfinite(d['lateral']).all()
        assert np.isin(d['y'], [0, 1]).all() and np.isin(d['mask'], [0, 1]).all()

# ------------------------------------------------------------------ outcomes
def outcome(p, y, mask):
    """Per-study outcome of answering autonomously at the fixed 0.5 cutoff (identical to V4-V13)."""
    fn = (y > .5) & (mask > .5) & (p < CUTOFF)
    fp = (y < .5) & (mask > .5) & (p >= CUTOFF)
    return dict(fn=(fn*SEVERITY).sum(1)/SEVERITY.sum(),
                severe=(fn[:, SEVERE].any(1)).astype(float),
                fp=fp.sum(1)/np.maximum((mask > .5).sum(1), 1))

# ------------------------------------------------------------------ classifier heads (numeric, cheap)
class Heads:
    """One standardiser + one regularised logistic regression per label."""
    def __init__(self, C=.1):
        self.C = C
    def fit(self, x, y, mask):
        self.scaler = StandardScaler().fit(x)
        z = self.scaler.transform(x)
        self.models = []
        for j in range(y.shape[1]):
            keep = mask[:, j] > .5
            yy = y[keep, j]
            if len(yy) < 20 or min(yy.sum(), len(yy)-yy.sum()) < 3:
                self.models.append(float((yy.sum()+1)/(len(yy)+2)))
            else:
                m = LogisticRegression(C=self.C, max_iter=300)
                m.fit(z[keep], yy)
                self.models.append(m)
        return self
    def predict(self, x):
        z = self.scaler.transform(x)
        return safe_prob(np.column_stack([np.full(len(z), m) if isinstance(m, float)
                                          else m.predict_proba(z)[:, 1] for m in self.models]))

def fit_temperature(p, y, mask, min_each=30):
    grid = np.geomspace(.25, 8, 61)
    def nll(pp, yy):
        q = safe_prob(expit(logit(safe_prob(pp))[:, None]/grid))
        return -(yy[:, None]*np.log(q)+(1-yy[:, None])*np.log1p(-q)).mean(0)
    keep = mask > .5
    tg = float(grid[np.argmin(nll(p[keep], y[keep]))]) if keep.sum() > 50 else 1.
    out = []
    for j in range(p.shape[1]):
        k = mask[:, j] > .5; yy = y[k, j]
        if min(yy.sum(), len(yy)-yy.sum()) < min_each:
            out.append(tg)
        else:
            out.append(float(grid[np.argmin(nll(p[k, j], yy))]))
    return np.array(out)

def apply_temperature(p, t):
    return safe_prob(expit(logit(safe_prob(p))/t))

def mean_brier(p, y, mask):
    return float((((p-y)**2)*mask).sum()/max(mask.sum(), 1))

def blend(pi, pt, present, w):
    return np.where(present[:, None], (1-w)*pi+w*pt, pi)

# ------------------------------------------------------------------ conformal (Mondrian / class-conditional)
def conformal_q(p, y, mask, alpha):
    q = np.ones((p.shape[1], 2))
    for j in range(p.shape[1]):
        for cls in [0, 1]:
            k = (mask[:, j] > .5) & (y[:, j] == cls)
            score = p[k, j] if cls == 0 else 1-p[k, j]
            n = int(k.sum()); rank = int(np.ceil((n+1)*(1-alpha)))
            if n and rank <= n:
                q[j, cls] = np.sort(score)[rank-1]
    return q

def pred_sets(p, q):
    return np.stack([p <= q[:, 0], (1-p) <= q[:, 1]], axis=-1)

def wilson(successes, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    p = successes/n; den = 1+z*z/n
    mid = (p+z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return float(mid-half), float(mid+half)

# ------------------------------------------------------------------ the five signals
def kl(a, b):
    return a*np.log(a/b)+(1-a)*np.log((1-a)/(1-b))

def signal_blocks(p0, pi, pt, present, img, txt, aux, q):
    """U uncertainty | D cross-modal disagreement | M missing context | S clinical severity.
    Only information available BEFORE any extra data is requested. Workload (W) acts on the threshold, not here."""
    s = pred_sets(p0, q); amb = s.sum(2) == 2
    ent = -(p0*np.log(p0)+(1-p0)*np.log1p(-p0))
    U = np.column_stack([p0, ent.mean(1), amb.mean(1), (s.sum(2) == 0).mean(1), np.abs(p0-.5).min(1)])
    mid = (pi+pt)/2
    js = .5*(kl(pi, mid)+kl(pt, mid))
    pr = present.astype(float)
    D = np.column_stack([js.mean(1)*pr, js.max(1)*pr, ((pi >= .5) != (pt >= .5)).mean(1)*pr, pr])
    zi = aux['img_scaler'].transform(img)
    atyp_i = ((zi-aux['img_pca'].inverse_transform(aux['img_pca'].transform(zi)))**2).mean(1)
    zt = aux['txt_scaler'].transform(txt)
    atyp_t = ((zt-aux['txt_pca'].inverse_transform(aux['txt_pca'].transform(zt)))**2).mean(1)*pr
    M = np.column_stack([1-pr, atyp_i, atyp_t])
    S = np.column_stack([(p0*SEVERITY).sum(1)/SEVERITY.sum(), (amb*SEVERITY).max(1),
                         (p0*SEVERITY*(p0 < .5)).sum(1)/SEVERITY.sum(), (p0*SEVERITY).max(1)])
    return dict(U=U, D=D, M=M, S=S)

GATE_SETS = {'U': ['U'], 'UD': ['U', 'D'], 'UDM': ['U', 'D', 'M'], 'UDMS': ['U', 'D', 'M', 'S']}
def design(blocks, groups):
    return np.concatenate([blocks[g] for g in groups], axis=1)

def hgb(seed, n_iter=80):
    return HistGradientBoostingRegressor(max_iter=n_iter, max_leaf_nodes=7, min_samples_leaf=20,
                                         l2_regularization=2., learning_rate=.08, random_state=seed)

def patient_weights(patients):
    _, inv, counts = np.unique(patients, return_inverse=True, return_counts=True)
    w = 1/counts[inv]
    return w/w.mean()

# ------------------------------------------------------------------ guarantees: CRC and Learn-then-Test (Hoeffding-Bentkus)
def hb_pvalue(risk_hat, n, alpha):
    """Valid p-value for H0: risk > alpha, for losses bounded in [0,1] (Angelopoulos et al., Learn-then-Test)."""
    r = np.minimum(risk_hat, alpha)
    h1 = xlogy(r, r/alpha)+xlogy(1-r, (1-r)/(1-alpha))
    hoeffding = np.exp(-n*h1)
    bentkus = np.e*binom.cdf(np.ceil(n*risk_hat-1e-12), n, alpha)
    return np.minimum(1., np.minimum(hoeffding, bentkus))

def ltt_index(L, alpha, delta):
    """L: n x G calibration losses, columns ordered conservative -> aggressive. Fixed-sequence testing.
    Returns the last column of the unbroken run of rejections, or -1 if even the first fails."""
    n = L.shape[0]
    ok = hb_pvalue(L.mean(0), n, alpha) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

def crc_index(L, alpha, B=1.0):
    """Conformal risk control for monotone families: largest aggressive column with (n*R+B)/(n+1) <= alpha."""
    n = L.shape[0]
    ok = (L.sum(0)+B)/(n+1) <= alpha
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

ESC_RATES = np.linspace(1, 0, 201)          # score gates: escalation fraction grid, conservative -> aggressive

def gate_thresholds(score_cal):
    t = np.quantile(score_cal, 1-ESC_RATES)
    t[0] = -np.inf
    return t

def gate_loss_matrix(score, harm, thresholds):
    return harm[:, None]*(score[:, None] < thresholds[None, :])

def calibrate_gate(score_cal, harm_cal, alpha, delta, method='ltt'):
    t = gate_thresholds(score_cal)
    L = gate_loss_matrix(score_cal, harm_cal, t)
    idx = ltt_index(L, alpha, delta) if method == 'ltt' else crc_index(L, alpha)
    return (-np.inf if idx < 0 else float(t[idx])), idx

# ------------------------------------------------------------------ 3-action routing (answer / escalate / ask)
def route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask):
    c0 = np.maximum(c0hat, 0.)
    c1 = np.maximum(c0hat-gainhat, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(allow_ask & eligible, (1-f)*c1+f*e*c0+lam*(a+f), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask

def realised(act, c0, c1, e, f):
    return np.where(act == 0, c0, np.where(act == 1, e*c0, (1-f)*c1+f*e*c0))

def workload_of(act, a, f):
    return np.where(act == 1, 1., np.where(act == 2, a+f, 0.))

def pick_lambda(c0hat, gainhat, eligible, e, f, a, allow_ask, budget, lam_grid):
    """Smallest price of workload whose label-free workload fits the budget."""
    for lam in lam_grid:
        act = route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask)
        if workload_of(act, a, f).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def topk_escalate(score, budget):
    n = len(score); k = int(np.floor(budget*n+1e-9))
    act = np.zeros(n, int)
    if k > 0:
        act[np.argsort(-score, kind='stable')[:k]] = 1
    return act

# ------------------------------------------------------------------ statistics
def boot_ci(x, reps=2000, seed=0):
    """Percentile 95% interval of the mean, a two-sided bootstrap p-value and the per-unit sd (for power calculations)."""
    x = np.asarray(x, float); n = len(x); rng = np.random.default_rng(seed)
    vals = np.empty(reps)
    for s in range(0, reps, 200):
        m = min(200, reps-s)
        vals[s:s+m] = x[rng.integers(0, n, (m, n))].mean(1)
    lo, hi = np.quantile(vals, [.025, .975])
    p = min(1., 2*(min((vals <= 0).sum(), (vals >= 0).sum())+1)/(reps+1))
    sd = float(x.std(ddof=1)) if n > 1 else float('nan')
    return dict(mean=float(x.mean()), lo=float(lo), hi=float(hi), n=n, p=float(p), sd=sd, mde80=float(2.8*sd/np.sqrt(n)) if n > 1 else float('nan'))

def holm(pvals):
    """Holm-Bonferroni adjusted p-values (family-wise error control)."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.
    for rank, i in enumerate(order):
        run = max(run, min(1., (m-rank)*p[i])); adj[i] = run
    return adj

def verdict(ci, lower_is_better=True):
    if ci['hi'] < 0:
        return 'better' if lower_is_better else 'worse'
    if ci['lo'] > 0:
        return 'worse' if lower_is_better else 'better'
    return 'inconclusive'

def auc_delta_ci(y, a, b, reps=1000, seed=0):
    y = np.asarray(y); n = len(y); rng = np.random.default_rng(seed); vals = []
    for _ in range(reps):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2:
            continue
        vals.append([roc_auc_score(y[i], b[i])-roc_auc_score(y[i], a[i]),
                     average_precision_score(y[i], b[i])-average_precision_score(y[i], a[i])])
    vals = np.asarray(vals)
    out = {}
    for j, (name, fn) in enumerate([('auroc', roc_auc_score), ('auprc', average_precision_score)]):
        lo, hi = np.quantile(vals[:, j], [.025, .975])
        out[name] = dict(a=float(fn(y, a)), b=float(fn(y, b)), delta=float(fn(y, b)-fn(y, a)), lo=float(lo), hi=float(hi))
    return out


# ------------------------------------------------------------------ four-action routing: answer / escalate / ask-context / ask-lateral
def route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p):
    c0 = np.maximum(c0hat, 0.)
    cc = np.maximum(c0-gc, 0.); cl = np.maximum(c0-gl, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(p['allow_ctx'] & ctx_ok, (1-p['f_c'])*cc+p['f_c']*e*c0+lam*(p['a_c']+p['f_c']), np.inf),
                    np.where(p['allow_lat'] & lat_ok, (1-p['f_l'])*cl+p['f_l']*e*c0+lam*(p['a_l']+p['f_l']), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask context, 3 ask lateral

def realised4(act, c0, cc, cl, e, p):
    return np.select([act == 0, act == 1, act == 2, act == 3],
                     [c0, e*c0, (1-p['f_c'])*cc+p['f_c']*e*c0, (1-p['f_l'])*cl+p['f_l']*e*c0])

def workload4(act, p):
    return np.select([act == 1, act == 2, act == 3], [1., p['a_c']+p['f_c'], p['a_l']+p['f_l']], 0.)

def pick_lambda4(c0hat, gc, gl, ctx_ok, lat_ok, e, p, budget, lam_grid):
    for lam in lam_grid:
        act = route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p)
        if workload4(act, p).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def hash_uniform(strings, salt):
    return np.array([int(hashlib.sha256(f'{salt}:{s}'.encode()).hexdigest()[:12], 16)/16**12 for s in strings])


# ------------------------------------------------------------------ learning-to-defer baseline (surrogate-loss gate)
from scipy.optimize import minimize
class L2DGate:
    """Learning-to-defer baseline in the spirit of Madras et al. / Mozannar & Sontag: a gate g(x) trained to minimise the expected team cost
    E[(1-g)*c + g*(e*c + lam)] for a fixed classifier and an expert that leaves a residual e of the loss. It is trained directly on the
    deferral objective (no intermediate loss regression). To give it a fair function class it uses all degree-2 feature interactions."""
    def fit(self, X, c, e, lam, w=None, reg=3e-3):
        from sklearn.preprocessing import PolynomialFeatures
        self.poly = PolynomialFeatures(2, include_bias=False)
        Z0 = self.poly.fit_transform((X-X.mean(0))/(X.std(0)+1e-8)); self.mu0, self.sd0 = X.mean(0), X.std(0)+1e-8
        self.mu = Z0.mean(0); self.sd = Z0.std(0)+1e-8; Z = (Z0-self.mu)/self.sd
        w = np.ones(len(X)) if w is None else w/np.mean(w)
        delta = (e*c+lam)-c
        def f(theta):
            g = expit(Z@theta[:-1]+theta[-1])
            loss = np.mean(w*(c+g*delta))+reg*np.sum(theta[:-1]**2)
            dz = w*g*(1-g)*delta/len(Z)
            return loss, np.concatenate([Z.T@dz+2*reg*theta[:-1], [dz.sum()]])
        self.theta = minimize(f, np.zeros(Z.shape[1]+1), jac=True, method='L-BFGS-B', options=dict(maxiter=400)).x
        return self
    def score(self, X):
        Z = (self.poly.transform((X-self.mu0)/self.sd0)-self.mu)/self.sd
        return Z@self.theta[:-1]+self.theta[-1]


def multi_ltt_index(Ls, alphas, delta):
    """Fixed-sequence Learn-then-Test for SEVERAL risks at once: a column is accepted only if every risk's Hoeffding-Bentkus p-value is <= delta
    (delta already includes the Bonferroni split across risks). Returns the last column of the unbroken accepted run, or -1."""
    n = Ls[0].shape[0]
    ok = np.ones(Ls[0].shape[1], bool)
    for L, a in zip(Ls, alphas):
        ok &= hb_pvalue(L.mean(0), n, a) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else Ls[0].shape[1]-1


## Load checkpoints

In [ ]:
K = PLAN['folds']
_cands = sorted((RUN_DIR/'methodology_v19').glob('*/seed0/fold0.npz'), key=lambda p: p.stat().st_mtime)
if not _cands:
    raise FileNotFoundError('No V19 fold checkpoints found under '+str(RUN_DIR/'methodology_v19')+'. Run the main V19 notebook first (it writes seed0/fold*.npz).')
OUT = _cands[-1].parent
print('Using fold checkpoints from', OUT)
BASE_KEYS = ['case', 'patient', 'eligible', 'present', 'y', 'mask', 'pr_img', 'pr_ic', 'pr_il', 'pr_icl']
SCHEME_KEYS = ['avail', 'ctx_ok', 'lat_ok', 'p0', 'dmean', 'fn0', 'fp0', 'sev0', 'fnc', 'fpc', 'sevc', 'fnl', 'fpl', 'sevl',
               'gc', 'gl', 'sc_entropy', 'sc_ambiguity', 'sc_sevharm', 'sc_l2d_U', 'sc_l2d_UDMS'] +               [f'{a}_{F}' for F in GATE_SETS for a in ['sc_fn', 'sc_sv', 'c0hat']] + \
              [f'miss_{l}' for l in PLAN['crit_labels']] + [f'scj_{l}_{F}' for l in PLAN['crit_labels'] for F in ['U', 'UDMS']]
def pooled(role, key):
    return np.concatenate([F[f'{role}_{key}'] for F in FOLDS])
def load_seed():
    """(Re)bind the pooled out-of-sample arrays of the CURRENT seed's folds (global OUT)."""
    global FOLDS, EVB, NE, FS
    FOLDS = [read_npz(OUT/f'fold{k}.npz') for k in range(K)]
    EVB = {key: pooled('ev', key) for key in BASE_KEYS}
    EVB['fold'] = np.concatenate([np.full(len(F['ev_case']), k) for k, F in enumerate(FOLDS)])
    NE = len(EVB['case'])
    assert len(set(EVB['patient'])) == NE, 'each evaluation patient must appear exactly once'
    FS = [np.flatnonzero(EVB['fold'] == k) for k in range(K)]
load_seed()
def build_EV(scheme):
    ev = dict(EVB)
    ev.update({key: pooled('ev', f'{scheme}_{key}') for key in SCHEME_KEYS})
    return ev
def get_cal(k, scheme):
    F = FOLDS[k]
    return {key: F[f'cal_{scheme}_{key}'] for key in SCHEME_KEYS}
PRIMARY = PLAN['primary_scheme']
EV = build_EV(PRIMARY)
RESULTS = {}      # everything that ends up in the report
TABLES = {}       # DataFrames exported to Excel/CSV
FPW = PLAN['fp_weight']
print(f'{NE:,} evaluation patients loaded from the checkpoints.')


## Recompute escalation decisions

In [ ]:
# Recompute only the escalation decisions needed for the audit (calibration only; no model training).
cals = [get_cal(k, PRIMARY) for k in range(K)]
runs = {}
for kind, alphas in PLAN['harm_kinds'].items():
    h0 = 'fn0' if kind == 'fn' else 'sev0'; pre = 'sc_fn_' if kind == 'fn' else 'sc_sv_'
    for alpha in alphas:
        for gate in ['U', 'UDMS']:
            esc = np.zeros(NE, bool)
            for k in range(K):
                t, _ = calibrate_gate(cals[k][pre+gate], cals[k][h0], alpha, PLAN['delta'], 'ltt')
                esc[FS[k]] = EV[pre+gate][FS[k]] >= t
            runs[(kind, alpha, gate, 'ltt')] = dict(esc=esc)
print('Escalation decisions recomputed for', len(runs), 'configurations.')


## Provide the MIMIC-IV tables (only if they are not already on Drive)
If the Drive folder does not exist, simply upload patients.csv.gz and admissions.csv.gz when this cell asks.

In [ ]:
# ---------------------------------------------------------------- optional: upload the MIMIC-IV tables straight into this Colab session
# Use this if you do not want to create the Drive folder. Run the cell, click "Choose files" and select patients.csv.gz (and admissions.csv.gz).
# Nothing is stored on Drive; the files live only in this session. If the tables are already in Drive or MIMIC_IV_DIR, the cell does nothing.
_already = (RUN_DIR/'mimic_iv'/'patients.csv.gz').is_file() or (globals().get('MIMIC_IV_DIR') and (Path(MIMIC_IV_DIR)/'patients.csv.gz').is_file())
if not _already and 'google.colab' in sys.modules:
    from google.colab import files as _files
    print('MIMIC-IV tables not found on Drive. Select patients.csv.gz (and admissions.csv.gz) to upload for this session:')
    _up = _files.upload()
    _dest = Path('/content/mimic_iv'); _dest.mkdir(exist_ok=True)
    for _name, _data in _up.items():
        _base = _name.split(' (')[0]                       # Colab renames duplicates "file (1).csv.gz"
        (_dest/(_base if _base.endswith(('.csv.gz', '.csv')) else _name)).write_bytes(_data)
    if (_dest/'patients.csv.gz').is_file() or (_dest/'patients.csv').is_file():
        MIMIC_IV_DIR = str(_dest); print('Using uploaded tables in', _dest)
elif _already:
    print('MIMIC-IV tables already available; no upload needed.')
else:
    print('Not running in Colab: place the tables in', RUN_DIR/'mimic_iv', 'or set MIMIC_IV_DIR.')


## Subgroup audit

In [ ]:
# ---------------------------------------------------------------- subgroup audit on the MIMIC cohort (needs MIMIC-IV; skipped otherwise)
# Demographics are NOT in MIMIC-CXR-JPG. With credentialed MIMIC-IV access, put hosp/patients.csv.gz (and optionally hosp/admissions.csv.gz) on Drive,
# e.g. <run folder>/mimic_iv/patients.csv.gz, or set MIMIC_IV_DIR to the folder holding them. We then ask whether the observed risk targets and the
# escalation burden are equitable across sex, age band and (if admissions are present) recorded race/ethnicity. Age is MIMIC-IV's anchor_age
# (age in the anchor year), an approximation of age at the study.
MIMIC_IV_DIR = globals().get('MIMIC_IV_DIR')      # set by the upload cell or by you
def mimic_subgroups():
    cfg_run = json.loads((RUN_DIR/'config.json').read_text())
    def mapped(value):
        for prefix in ['/content/drive_fresh/MyDrive/', '/content/drive/MyDrive/']:
            if str(value).startswith(prefix):
                return RUN_DIR.parents[2]/str(value)[len(prefix):]
        return Path(value)
    jroot = mapped(cfg_run['jpg_root'])
    dirs = [Path(MIMIC_IV_DIR)] if MIMIC_IV_DIR else []
    dirs += [RUN_DIR/'mimic_iv', RUN_DIR.parent/'mimic_iv', jroot.parent/'mimic-iv'/'hosp', jroot.parent/'mimic_iv', jroot/'mimic_iv']
    found = [d for d in dirs if (d/'patients.csv.gz').is_file() or (d/'patients.csv').is_file()]
    if not found:
        # last resort: look for the MIMIC-IV tables anywhere under the project folders on Drive (shallow, 4 levels)
        for root in [RUN_DIR, RUN_DIR.parent, RUN_DIR.parents[1]]:
            if not root.is_dir():
                continue
            base_depth = len(root.parts)
            for cur, subdirs, fnames in os.walk(root):
                if len(Path(cur).parts)-base_depth >= 4:
                    subdirs[:] = []
                if any(f in ('patients.csv.gz', 'patients.csv') for f in fnames):
                    found.append(Path(cur)); break
            if found:
                break
    if not found:
        listing = sorted(p.name for p in (RUN_DIR/'mimic_iv').iterdir()) if (RUN_DIR/'mimic_iv').is_dir() else 'folder does not exist'
        raise FileNotFoundError('MIMIC-IV patients table not found. Looked in: '+'; '.join(str(x) for x in dirs)+' and below the project folders. '
                                'Contents of '+str(RUN_DIR/'mimic_iv')+': '+str(listing)+'. Put patients.csv.gz there (exact name) or set MIMIC_IV_DIR.')
    print('Using MIMIC-IV tables from', found[0])
    d = found[0]; pf = d/'patients.csv.gz' if (d/'patients.csv.gz').is_file() else d/'patients.csv'
    pts = pd.read_csv(pf, dtype={'subject_id': str})[['subject_id', 'gender', 'anchor_age']]
    cohort = pd.read_csv(RUN_DIR/'cohort_PRIVATE.csv', dtype=str)
    cohort['case'] = [hashlib.sha256(f"{cfg_run['seed']}:{p}:{s}".encode()).hexdigest()[:20] for p, s in zip(cohort.subject_id, cohort.study_id)]
    meta = cohort[['case', 'subject_id']].drop_duplicates('case').merge(pts, on='subject_id', how='left').set_index('case')
    af = d/'admissions.csv.gz' if (d/'admissions.csv.gz').is_file() else (d/'admissions.csv' if (d/'admissions.csv').is_file() else None)
    if af is not None:
        adm = pd.read_csv(af, dtype={'subject_id': str})[['subject_id', 'race']].dropna()
        adm['race_group'] = adm.race.str.upper().map(lambda r: 'WHITE' if r.startswith('WHITE') else 'BLACK' if 'BLACK' in r else 'ASIAN' if 'ASIAN' in r else 'HISPANIC' if 'HISPANIC' in r else 'OTHER/UNKNOWN')
        race = adm.groupby('subject_id').race_group.agg(lambda s: s.value_counts().index[0])
        meta['race_group'] = meta.subject_id.map(race)
    ev = EV; m_ = meta.reindex(ev['case'])
    age = m_.anchor_age.to_numpy(float)
    groups = {'sex=F': (m_.gender == 'F').to_numpy(), 'sex=M': (m_.gender == 'M').to_numpy(), 'age<40': age < 40, 'age 40-64': (age >= 40) & (age < 65), 'age>=65': age >= 65}
    if 'race_group' in m_:
        for g in ['WHITE', 'BLACK', 'ASIAN', 'HISPANIC', 'OTHER/UNKNOWN']:
            groups['race='+g] = (m_.race_group == g).to_numpy()
    rows = []
    for kind, h0 in [('fn', 'fn0'), ('severe', 'sev0')]:
        for alpha in PLAN['harm_kinds'][kind]:
            for gate in ['U', 'UDMS']:
                esc = runs[(kind, alpha, gate, 'ltt')]['esc']
                for name, mg in groups.items():
                    if mg.sum() < 100:
                        continue
                    loss = ev[h0][mg]*(~esc[mg]); ci = boot_ci(loss, PLAN['boot'], 76)
                    rows.append(dict(harm=kind, alpha=alpha, gate=gate, group=name, n=int(mg.sum()), answer_all_risk=float(ev[h0][mg].mean()), risk=ci['mean'], lo=ci['lo'], hi=ci['hi'],
                                     referral=float(esc[mg].mean()), point_target_met=bool(ci['mean'] <= alpha), upper_bound_met=bool(ci['hi'] <= alpha)))
    T = pd.DataFrame(rows)
    aur = []
    for name, mg in groups.items():
        if mg.sum() < 100:
            continue
        a_ = []
        for jj in range(len(LABELS)):
            k_ = mg & (ev['mask'][:, jj] > .5); yy = ev['y'][k_, jj]
            if min(yy.sum(), len(yy)-yy.sum()) >= 20:
                a_.append(roc_auc_score(yy, ev['p0'][k_, jj]))
        aur.append(dict(group=name, n=int(mg.sum()), macro_auroc=float(np.mean(a_)) if a_ else np.nan, labels_assessed=len(a_)))
    return T, pd.DataFrame(aur)

try:
    SGT, SGA = mimic_subgroups(); TABLES['mimic_subgroup_risk'] = SGT; TABLES['mimic_subgroup_auroc'] = SGA
    display(SGA.round(3)); display(SGT[(SGT.gate == 'UDMS')].round(4))
    full = SGT[SGT.gate == 'UDMS']
    RESULTS['mimic_subgroups'] = dict(groups=sorted(full.group.unique()), configs=int(len(full)), point_target_met=int(full.point_target_met.sum()),
                                      upper_bound_met=int(full.upper_bound_met.sum()), auroc_range=[float(SGA.macro_auroc.min()), float(SGA.macro_auroc.max())])
    print('MIMIC subgroup audit:', RESULTS['mimic_subgroups'])
except Exception as exc:
    print('MIMIC subgroup audit skipped:', type(exc).__name__, exc)


## Save and download

In [ ]:
import zipfile
out_dir = OUT/'subgroup_audit'; out_dir.mkdir(exist_ok=True)
for name, df in TABLES.items():
    df.to_csv(out_dir/f'{name}.csv', index=False)
json_write(out_dir/'mimic_subgroups.json', RESULTS.get('mimic_subgroups', {'status': 'skipped'}))
zp = out_dir/'V19_MIMIC_SUBGROUPS.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out_dir.glob('*')):
        if p.suffix in ('.csv', '.json'):
            z.write(p, p.name)
print('Saved', zp)
try:
    from google.colab import files
    files.download(str(zp))
except Exception:
    pass
